# LLM Forge on a free GPU (Kaggle / Google Colab)

No GPU of your own? Run this notebook on a free one:

* **Kaggle** (≈30 free GPU hours/week): *Settings → Accelerator → GPU T4 x2*, and *Settings → Internet → On*.
* **Colab**: *Runtime → Change runtime type → T4 GPU*.

A T4 has 16 GB: enough for Qwen3-0.6B/1.7B with `lora`, or Qwen3-4B with `qlora`.

Two ways to use it: **A)** train from notebook cells (simplest, most reliable), or **B)** open the full web UI through a temporary public link.

In [ ]:
# If the repo is private, use https://<YOUR_GITHUB_TOKEN>@github.com/Jamersop/llm-forge
!git clone -q https://github.com/Jamersop/llm-forge
%cd llm-forge
!pip install -q -r requirements.txt bitsandbytes
!python -m forge.cli info

## Your data
Upload `.jsonl` / `.csv` / `.txt` files (see the README for formats), then list them below.

* **Colab**: run the next cell to pick files.
* **Kaggle**: *Add Input → Upload dataset*; files appear under `/kaggle/input/<name>/`.

The example files in `examples/` are used by default so you can do a test run first.

In [ ]:
import os
DATA_FILES = ["examples/medical_triage_reasoning.jsonl", "examples/domain_corpus.txt"]

try:  # Colab upload widget
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        DATA_FILES = list(uploaded)
except ImportError:
    pass  # Kaggle: set DATA_FILES = ["/kaggle/input/<name>/<file>.jsonl", ...]
print(DATA_FILES)

## A) Train from the notebook
Gated models (Llama, Gemma) need a Hugging Face token: `os.environ["HF_TOKEN"] = "hf_..."`.

In [ ]:
BASE_MODEL = "Qwen/Qwen3-1.7B"   # Qwen3-4B -> also set METHOD = "qlora"
METHOD = "lora"
EPOCHS = 3

data_args = " ".join(f'--data "{f}"' for f in DATA_FILES)
!python -m forge.cli train --base-model {BASE_MODEL} {data_args} --out runs/my-model \
    --set method={METHOD} --set epochs={EPOCHS} --set batch_size=2 --set grad_accum=8 --set max_seq_len=2048

In [ ]:
# Try it
from forge.inference import load_weights, generate
model, tok = load_weights(BASE_MODEL, "runs/my-model/adapter", is_adapter=True)
r = generate(model, tok, [{"role": "user", "content": "An adult has crushing chest pain and is sweating. What first?"}],
             max_new_tokens=600)
print("THINKING:\n", r["thinking"], "\n\nANSWER:\n", r["answer"])

### Keep the result
Notebook storage is wiped when the session ends. Download the adapter (small, ~10–100 MB),
or push it to the Hugging Face Hub.

In [ ]:
!cd runs && zip -qr my-model-adapter.zip my-model/adapter && ls -lh my-model-adapter.zip
try:
    from google.colab import files; files.download("runs/my-model-adapter.zip")
except ImportError:
    print("Kaggle: find it in the Output panel (save a version), or push to the Hub below.")

# Optional: push to the Hugging Face Hub
# from huggingface_hub import HfApi
# HfApi(token="hf_...").create_repo("your-name/my-model-adapter", exist_ok=True)
# HfApi(token="hf_...").upload_folder(folder_path="runs/my-model/adapter", repo_id="your-name/my-model-adapter")

## B) Full web UI through a temporary public link
Starts the Forge server on this GPU and opens a Cloudflare quick tunnel. Open the printed
`https://….trycloudflare.com` link. **Anyone with the link can use it while it runs.**

In [ ]:
import subprocess, re, time
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
server = subprocess.Popen(["python", "-m", "uvicorn", "forge.api:app", "--port", "8000"],
                          stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    if m := re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line):
        print("Open:", m.group(0)); break